# Non-Linear Economy Estimation (ANN)

Please note that this notebook uses a venv which points to a base python version of **3.13**, some functionality may be limited if using an older version of python.

## All Imports

In [3]:
%pip install scikit-learn seaborn torch torchvision gymnasium stable-baselines3 --quiet

Note: you may need to restart the kernel to use updated packages.


In [4]:
%matplotlib widget

## Data From Source Package

In [5]:
%pip install -e ../ --quiet

Note: you may need to restart the kernel to use updated packages.


In [6]:
import autonomous_fed as afed

In [7]:
le_solver = afed.LinearEnvironmentSolver(fred_key="7ab121fb17773e187bb6508e83e411da")

# Data Check
print (le_solver.historical_data.head())
print (le_solver.historical_data.tail())

             pi         y     i
date                           
1987Q3  2.66122 -0.388640  6.84
1987Q4  2.91876  0.526730  6.92
1988Q1  3.06577  0.260731  6.66
1988Q2  3.35274  0.788853  7.16
1988Q3  3.80207  0.595530  7.98
             pi         y     i
date                           
2006Q2  3.35642  1.319357  4.91
2006Q3  3.13805  0.978513  5.25
2006Q4  2.66316  1.380249  5.25
2007Q1  2.91019  1.210046  5.26
2007Q2  2.72883  1.336951  5.25


## Model Architecture

### NARX Model

For our economy transition equations we have: ${y_t=\hat{f}^y(y_{t-1},y_{t-2},\pi_t,\pi_{t-1},\pi_{t-2},i_t,i_{t-1},i_{t-2})+\epsilon_t^y}$ and ${\pi_t=\hat{f}^\pi(y_t,y_{t-1},y_{t-2},\pi_{t-1},\pi_{t-2},i_t,i_{t-1},i_{t-2})+\epsilon_t^\pi}$

In this scenario our predictor function, ${\hat{f}}$ is an ANN (Artificial Neural Network) but it can be swapped with other nonlinear functions such as a sigmoid function or wavelet network. For our ANN predictor we have ${\hat{f}^m=b_0^m+\sum_{j=1}^h v_j^mG(\omega_j^{m'}s_t^m+b_j^m), m\in\{y,\pi}\}$

The Components of the ANN are as follows:
- ${m}$: ${\{y,\pi}\}$
- ${s_t^m}$: Input state vectors at time t.
- ${w_j^m}$: Weight vector for the j-th hidden neuron.
- ${b_j^m}$: Bias term for the current neuron.
- ${G(\cdot)}$: Activation function (nonlinear transform).
- ${v_j^m}$: Weight from hidden neuron ${j}$ to the output layer
- ${b_0^m}$: Bias at the output layer
- ${h}$: Number of hidden neurons.

As seen above the Neural Network type is a NARX Model with a single hidden layer and the activation function is the hyperbolic tangent therefore we define ${G(\cdot)}$ as follows: ${G(x)=tanh(x)=\frac{e^x-e^-x}{e^x+e^-x}}$

### Model Buildout

In [ ]:
from typing import List, Tuple
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

def make_lag_matrix(df: pd.DataFrame, target_col: str, lag_cols: List[str], p: int = 4, holdout_frac: float = 0.15) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, List[str], Tuple[pd.Series, pd.Series]]:
    """
    Create lagged design matrix for NARX-style modeling.

    Args:
        df (pd.DataFrame): DataFrame with time series data, sorted by time
        target_col (str): name of target column to predict
        lag_cols (List[str]): list of column names to use as inputs (will be lagged)
        p (int): number of lags to include
        holdout_frac (float): fraction of data to hold out for validation

    Returns:
        Tuple containing:
            Xt (torch.Tensor): training inputs
            yt (torch.Tensor): training targets
            Xv (torch.Tensor): validation inputs
            yv (torch.Tensor): validation targets
            feature_names (List[str]): names of features in design matrix
            scaler (Tuple[pd.Series, pd.Series]): (mean, std) used for standardization

    Raises:
        ValueError: if holdout_frac is not in (0, 1)
    """
    # Build lagged design matrix
    X_parts = []
    for col in lag_cols:
        for L in range(1, p+1):
            X_parts.append(df[col].shift(L).rename(f"{col}_lag{L}"))
    X = pd.concat(X_parts, axis=1)
    y = df[target_col].copy()

    # Drop initial p rows with NaNs from lagging
    X, y = X.iloc[p:], y.iloc[p:]

    # Train/val split (last 15% is validation)
    n = len(X)
    n_val = int(np.ceil(n * holdout_frac))
    split = n - n_val

    X_train, X_val = X.iloc[:split], X.iloc[split:]
    y_train, y_val = y.iloc[:split], y.iloc[split:]

    # Standardize inputs (fit on train, apply to val)
    mu, sigma = X_train.mean(), X_train.std().replace(0, 1.0)
    X_train_z = (X_train - mu) / sigma
    X_val_z   = (X_val   - mu) / sigma

    # Tensors
    Xt = torch.tensor(X_train_z.values, dtype=torch.float32)
    Xv = torch.tensor(X_val_z.values,   dtype=torch.float32)
    yt = torch.tensor(y_train.values,   dtype=torch.float32).view(-1, 1)
    yv = torch.tensor(y_val.values,     dtype=torch.float32).view(-1, 1)
    return Xt, yt, Xv, yv, X.columns.tolist(), (mu, sigma)

class Eq8Net(nn.Module):
    """
    Single hidden layer neural network with tanh activation, replicating equation (8).

    Attributes:
        hidden (nn.Linear): hidden layer
        act (nn.Tanh): activation function
        out (nn.Linear): output layer

    Methods:
        forward(x: torch.Tensor) -> torch.Tensor: forward pass through the network
    """
    def __init__(self, in_dim: int, h: int) -> None:
        """
        Initialize the Eq8Net model.

        Args:
            in_dim (int): input feature dimension
            h (int): number of hidden units
        
        Returns:
            None

        Raises:
            None
        """
        super().__init__()
        self.hidden: nn.Linear = nn.Linear(in_dim, h)      # ω_j
        self.act: nn.Tanh = nn.Tanh()                 # G(·) = tanh
        self.out: nn.Linear = nn.Linear(h, 1)           # ν_j and b0
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass through the network.

        Args:
            x (torch.Tensor): input tensor of shape (batch_size, in_dim)
        
        Returns:
            torch.Tensor: output tensor of shape (batch_size, 1)

        Raises:
            None
        """
        return self.out(self.act(self.hidden(x)))

def fit_once(Xt: torch.Tensor, yt: torch.Tensor, Xv: torch.Tensor, yv: torch.Tensor, h: int, seed: int, lr: float = 1e-3, batch: int = 128, max_epochs: int = 2000, patience: int = 40) -> Tuple[Eq8Net, float]:
    """
    Fit Eq8Net model once with given hyperparameters and random seed.

    Args:
        Xt (torch.Tensor): training inputs
        yt (torch.Tensor): training targets
        Xv (torch.Tensor): validation inputs
        yv (torch.Tensor): validation targets
        h (int): number of hidden units
        seed (int): random seed for reproducibility
        lr (float): learning rate for Adam optimizer
        batch (int): batch size for training
        max_epochs (int): maximum number of training epochs
        patience (int): epochs to wait for improvement before early stopping

    Returns:
        Tuple containing:
            model (Eq8Net): trained model with best validation performance
            best_val (float): best validation MSE achieved
    
    Raises:
        None
    """
    torch.manual_seed(seed)
    model = Eq8Net(Xt.shape[1], h)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    train_loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch, shuffle=True)

    best_state, best_val, wait = None, float('inf'), 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        # validation
        model.eval()
        with torch.no_grad():
            val = loss_fn(model(Xv), yv).item()
        if val + 1e-10 < best_val:
            best_val = val
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val

def fit_with_model_selection(Xt: torch.Tensor, yt: torch.Tensor, Xv: torch.Tensor, yv: torch.Tensor, h_grid: range = range(1, 11), restarts: int = 30) -> Tuple[Eq8Net, int, float]:
    """
    Fit Eq8Net model with hyperparameter selection over hidden units and random restarts.

    Args:
        Xt (torch.Tensor): training inputs
        yt (torch.Tensor): training targets
        Xv (torch.Tensor): validation inputs
        yv (torch.Tensor): validation targets
        h_grid (range): range of hidden units to try
        restarts (int): number of random restarts per hidden unit setting

    Returns:
        Tuple containing:
            best_model (Eq8Net): model with best validation performance
            best_h (int): number of hidden units for best model
            best_val (float): best validation MSE achieved
    
    Raises:
        None
    """
    best_model, best_h, best_val = None, None, float('inf')
    for h in h_grid:
        for seed in range(restarts):
            model, val = fit_once(Xt, yt, Xv, yv, h=h, seed=seed)
            if val < best_val:
                best_model, best_h, best_val = model, h, val
    return best_model, best_h, best_val

@torch.no_grad()
def dynamic_forecast(model: Eq8Net, df_last: pd.DataFrame, lag_cols: list, p: int, scaler, horizon: int, target_name: str, other_targets_updater: callable =None) -> np.ndarray:
    """
    Perform dynamic multi-step forecasting using the trained NARX-style model.

    Args:
        model (Eq8Net): trained NARX-style model
        df_last (pd.DataFrame): DataFrame with the most recent data to start forecasting from
        lag_cols (list): list of column names used as inputs (will be lagged)
        p (int): number of lags used in the model
        scaler (Tuple[pd.Series, pd.Series]): (mean, std) used for standard
        horizon (int): number of steps to forecast
        target_name (str): name of the target column being forecasted
        other_targets_updater (callable, optional): function to update other target columns after each forecast step

    Returns:
        np.ndarray: array of forecasted values of shape (horizon,)

    Raises:
        None
    """
    mu, sigma = scaler
    df_roll = df_last.copy().iloc[-p:].copy()
    preds = []
    for _ in range(horizon):
        # build one-step feature row
        X_parts = []
        for col in lag_cols:
            for L in range(1, p+1):
                X_parts.append(df_roll[col].iloc[-L])
        x = pd.Series(X_parts, index=[f"{c}_lag{L}" for c in lag_cols for L in range(1, p+1)])
        xz = ((x - mu) / sigma).fillna(0.0).values.astype(np.float32)
        xz = torch.tensor(xz).unsqueeze(0)
        yhat = model(xz).item()
        preds.append(yhat)
        # append prediction and advance window
        new_row = df_roll.iloc[-1].copy()
        new_row[target_name] = yhat
        df_roll = pd.concat([df_roll, new_row.to_frame().T], axis=0)
        if other_targets_updater:
            df_roll = other_targets_updater(df_roll)
    return np.array(preds)

In [12]:
df = (
    le_solver.historical_data
    .loc[:, ["y", "pi", "i"]]    # add any other exogenous series you used
    .dropna()
    .sort_index()
)
p = 4                  # number of lags (match the linear model)
lag_cols = ["y","pi","i"] # state vector used in the paper

# --- y model ---
Xt_y, yt_y, Xv_y, yv_y, cols_y, scaler_y = make_lag_matrix(
    df, target_col="y", lag_cols=lag_cols, p=p, holdout_frac=0.15
)
model_y, h_y, val_y = fit_with_model_selection(Xt_y, yt_y, Xv_y, yv_y)
print(f"Best h (y)={h_y}  Val MSE (y)={val_y:.6f}")

# --- pi model ---
Xt_p, yt_p, Xv_p, yv_p, cols_p, scaler_p = make_lag_matrix(
    df, target_col="pi", lag_cols=lag_cols, p=p, holdout_frac=0.15
)
model_p, h_p, val_p = fit_with_model_selection(Xt_p, yt_p, Xv_p, yv_p)
print(f"Best h (pi)={h_p}  Val MSE (pi)={val_p:.6f}")

import torch.nn as nn
mse = nn.MSELoss()

with torch.no_grad():
    train_mse_y = mse(model_y(Xt_y), yt_y).item()
    val_mse_y   = mse(model_y(Xv_y), yv_y).item()
    train_mse_p = mse(model_p(Xt_p), yt_p).item()
    val_mse_p   = mse(model_p(Xv_p), yv_p).item()

print(f"[y]  train {train_mse_y:.6f}  val {val_mse_y:.6f}")
print(f"[pi] train {train_mse_p:.6f}  val {val_mse_p:.6f}")

import pickle, pathlib, torch

artifacts_dir = pathlib.Path("artifacts_ann_eq8")
artifacts_dir.mkdir(exist_ok=True)

torch.save(model_y.state_dict(), artifacts_dir/"model_y.pt")
torch.save(model_p.state_dict(), artifacts_dir/"model_p.pt")

with open(artifacts_dir/"features.pkl", "wb") as f:
    pickle.dump(
        dict(cols_y=cols_y, cols_p=cols_p, scaler_y=scaler_y, scaler_p=scaler_p, p=p, lag_cols=lag_cols),
        f
    )

# use the last p rows for lag construction
df_last = df.iloc[-p:].copy()

# forecast y 8 steps ahead, keeping pi and i fixed to last observed values
y_fc = dynamic_forecast(
    model=model_y,
    df_last=df_last,
    lag_cols=lag_cols,
    p=p,
    scaler=scaler_y,
    horizon=8,
    target_name="y",
    other_targets_updater=None,   # nothing else changes each step
)
print(y_fc)

def joint_updater_factory(latest_pred_dict):
    def _upd(df_roll):
        # write back any other targets that were predicted this step
        for k, v in latest_pred_dict.items():
            df_roll.iloc[-1, df_roll.columns.get_loc(k)] = v
        return df_roll
    return _upd

# example: at each step, predict y first, then pi, and feed both back
horizon = 8
df_roll = df.iloc[-p:].copy()
mu_y, sig_y = scaler_y
mu_p, sig_p = scaler_p
y_preds, p_preds = [], []

for _ in range(horizon):
    # predict y
    yhat = dynamic_forecast(model_y, df_roll, lag_cols, p, scaler_y, horizon=1, target_name="y")[0]
    # predict pi with yhat fed back
    latest_map = {"y": yhat}
    updater = joint_updater_factory(latest_map)
    pihat = dynamic_forecast(model_p, df_roll, lag_cols, p, scaler_p, horizon=1, target_name="pi",
                             other_targets_updater=updater)[0]
    # commit both to df_roll end row and advance one step
    new_row = df_roll.iloc[-1].copy()
    new_row["y"], new_row["pi"] = yhat, pihat
    df_roll = pd.concat([df_roll, new_row.to_frame().T], axis=0)

    y_preds.append(yhat)
    p_preds.append(pihat)

print("y preds:", y_preds)
print("pi preds:", p_preds)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Move existing models to device
model_y = model_y.to(device)
model_p = model_p.to(device)

print("Models moved to", device)

Best h (y)=1  Val MSE (y)=0.131344
Best h (pi)=9  Val MSE (pi)=0.026406
[y]  train 1.632311  val 0.131344
[pi] train 0.019909  val 0.026406
[1.16285396 1.2187978  1.19207048 1.2322396  1.21012104 1.21579194
 1.21559751 1.22065425]
y preds: [np.float64(1.1628539562225342), np.float64(1.2204439640045166), np.float64(1.1732256412506104), np.float64(1.2084333896636963), np.float64(1.2358222007751465), np.float64(1.2852816581726074), np.float64(1.3138989210128784), np.float64(1.352474570274353)]
pi preds: [np.float64(2.599834442138672), np.float64(2.4061810970306396), np.float64(2.2914085388183594), np.float64(2.165032148361206), np.float64(2.0526955127716064), np.float64(1.964349389076233), np.float64(1.9064844846725464), np.float64(1.8722132444381714)]
Using device: cpu
Models moved to cpu
